# JavaScript — Forms & dynamic lists

> **How this topic works**
> 1. **This notebook** — read the theory.
> 2. **`project/`** — a real Vite app where you do the exercise in the browser.
>
> Read this first, then follow the steps at the bottom. DOM snippets are shown as
> plain code blocks (this kernel has no browser); runnable cells are marked.

## LESSON 46 — Forms and dynamic lists

This is the shape of almost every real app screen:

```
    DATA  →  render()  →  the page
      ↑                      │
      └──── an event ────────┘
```

1. some **data** lives in a variable
2. a **render** function draws that data into the page
3. events **change the data**, then call render again

Never patch the page by hand in ten different places. Change the data, re-render. **This is how React thinks**, minus the framework — get it here and React becomes a syntax change rather than a new way of thinking.

### Forms

```js
form.addEventListener("submit", (event) => {
  event.preventDefault();          // or the page reloads
  const value = input.value.trim();
  // ...validate, update data, re-render
  input.value = "";                // clear the field
});
```

### Rendering a list

```js
function render() {
  list.innerHTML = "";                       // clear, then rebuild

  todos.forEach((todo, index) => {
    const item = document.createElement("li");
    item.textContent = todo;
    item.dataset.index = index;              // remember which one this is
    list.append(item);
  });
}
```

`dataset.index` writes a `data-index` attribute, read back with `event.target.dataset.index` — always as a **string**, so wrap it in `Number()`.

### Key notes

- **Listen for `submit` on the form, not `click` on the button.** Only `submit` catches the user pressing Enter in the field.
- **Always `event.preventDefault()`.** Without it the page reloads and every variable your script held is gone.
- `input.value` is always a **string**, even with `type="number"`.
- Rebuilding the whole list on every change feels wasteful and is exactly right at this scale. It also fixes stale indexes for free.

### The data → render loop — runnable

The loop has nothing to do with the DOM, so you can watch it here. `render()`
returns text instead of touching a page; in `project/` it creates `<li>` elements.
Notice that **nothing ever edits the output directly** — every change goes through
the data.

In [ ]:
const todos = [];

function render() {
  if (todos.length === 0) return "(empty)";
  return todos.map((todo, index) => `${index}: ${todo}`).join("\n");
}

function addTodo(text) {
  const value = text.trim();
  if (value === "") return "Type something first.";  // validation
  todos.push(value);                                  // change the DATA
  return render();                                    // then re-render
}

function removeTodo(index) {
  todos.splice(index, 1);
  return render();
}

console.log(render());
console.log("--");
console.log(addTodo("buy bread"));
console.log("--");
console.log(addTodo("  "));        // rejected
console.log("--");
console.log(addTodo("call Mia"));
console.log("--");
console.log(removeTodo(0));        // indexes shift — re-rendering handles it

## LESSON 47 — Rendering without rebuilding

LESSON 46 rebuilds the whole list on every change. That is the right default: one code path, no chance of the page and the data disagreeing.

It has a cost, though. Emptying a `<ul>` throws away every element inside it — along with which one had focus, where the text selection was, and any scroll position. On a long list the user notices.

### insertAdjacentHTML — adding without replacing

```js
list.insertAdjacentHTML("beforeend", `<li>${text}</li>`);
```

| position | where the HTML lands |
|---|---|
| `"beforebegin"` | just before the element |
| `"afterbegin"` | inside, as the first child |
| `"beforeend"` | inside, as the last child |
| `"afterend"` | just after the element |

It parses a string into real elements, so it is faster than building each node by hand, and it leaves the existing children alone.

The catch is the same one `innerHTML` has: a string that came from a user can carry markup. Never interpolate text you did not create — use `textContent` for that.

### template — markup kept out of your JavaScript

A `<template>` holds markup the browser parses but does not display, so a row's shape stays in the HTML where it belongs.

```html
<template id="row">
  <li class="task"><span class="label"></span></li>
</template>
```

```js
const template = document.querySelector("#row");

function makeRow(text) {
  const row = template.content.cloneNode(true);   // true = copy the children too
  row.querySelector(".label").textContent = text;
  return row;
}

list.append(makeRow("Buy milk"));
```

`cloneNode(true)` is what you append — never `template.content` itself, or the second call finds an empty template.

### Which to use

Rebuild everything until it is visibly a problem. Then add rows with `insertAdjacentHTML` or a `<template>` and keep removal by delegation. Correct first, fast second.

### Key notes

- **`cloneNode(true)` copies the children; `cloneNode()` alone copies only the outer element.** Forget the `true` and you append an empty shell.
- **`insertAdjacentHTML` parses HTML.** User text goes in with `textContent`, never inside the string.
- `"beforeend"` and `"afterbegin"` are inside the element; `"beforebegin"` and `"afterend"` are outside it. The four names are worth reading twice.
- Appending a template's content **moves** it. That is why you clone.

### The four insert positions — runnable

No browser here, so this cell models what each position does to a list of children.

In [ ]:
// A tiny stand-in for a parent element and its children.
const children = ["a", "b"];

function insertAdjacent(position, value) {
  const copy = [...children];
  if (position === "afterbegin") copy.unshift(value);
  if (position === "beforeend") copy.push(value);
  return copy;
}

console.log("afterbegin ->", insertAdjacent("afterbegin", "NEW"));
console.log("beforeend  ->", insertAdjacent("beforeend", "NEW"));

// beforebegin and afterend land OUTSIDE the element, so they never touch
// this array — they would sit next to the parent itself.
console.log("children are untouched:", children);

---

## Now do the exercise

**1. Start the project**

```bash
cd project
npm install     # only the first time
npm run dev
```

**2. Read the live demo**

Open `project/src/lessons/lesson-46-forms-lists.js`. It builds the *colours* box
with exactly the loop above. Add a colour, add a duplicate, click one to remove it.

**3. Do the exercise**

Open `project/src/exercise/exercise.js` and work through the numbered STEPs.
You're rebuilding the same machine for a to-do list.

**Done when:** you can add tasks, an empty submit is refused with a message, and
clicking a task removes it — including after several adds and removes.

Stuck? Paste `ai-prompt.txt` into a fresh AI session. The answer is in
`project/src/exercise/solution.js` — last resort.